# Multi-illumination robustness

PKU-Market-PCB filenames carry a `light_<NN>` token. This notebook evaluates the published baseline and P2 checkpoints on each lighting subset (no retraining) and writes metrics plus plots to `illumination_outputs/`.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.chdir('/content/drive/MyDrive/lld-net-pcb-ml')
except ImportError:
    pass

from project_paths import setup, prepare_dataset

P = setup()
os.chdir(P.repo_root)
print('repo :', P.repo_root)
print('colab:', P.in_colab)

In [ ]:
import shutil

if shutil.which('nvidia-smi'):
    !nvidia-smi
else:
    print('nvidia-smi not found — CPU / no NVIDIA driver.')

try:
    import google.colab
    %pip -q install ultralytics pandas pyyaml opencv-python matplotlib seaborn
except ImportError:
    pass

In [ ]:
import os
import re
import json
import shutil
from pathlib import Path
from collections import defaultdict

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import ultralytics

sns.set_theme(context='talk', style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Ultralytics:', ultralytics.__version__)

In [ ]:
REPO_ROOT          = P.repo_root
DRIVE_DATASET_DIR  = P.drive_dataset
DRIVE_RUNS_DIR     = P.drive_runs
OUT_DIR            = P.drive_illum
(OUT_DIR / 'plots').mkdir(parents=True, exist_ok=True)

P2_AUG_PROFILE = 'strong'
BASELINE_BEST  = P.baseline_best
P2_BEST        = P.p2_best
if not P2_BEST.exists():
    legacy = DRIVE_RUNS_DIR / 'yolo12n_p2' / 'weights' / 'best.pt'
    if legacy.exists():
        print(f'[warn] using legacy P2 folder: {legacy}')
        P2_BEST = legacy

LOCAL_DATASET_DIR  = P.local_dataset
LOCAL_DATA_YAML    = P.local_data_yaml
LOCAL_ILLUM_ROOT   = P.local_illum

IMG_SIZE = 640
DEVICE   = 0 if __import__('torch').cuda.is_available() else 'cpu'

for f in (BASELINE_BEST, P2_BEST):
    if not f.exists():
        raise FileNotFoundError(f'missing: {f}')

In [ ]:
yml = prepare_dataset(P)
CLASSES = yml['names']
print('Classes:', CLASSES)

In [ ]:
# Bucket the test split by parsing the `light_<NN>` token from each filename.
LIGHT_RE = re.compile(r'^(?:l_)?light_(\d+)_', re.IGNORECASE)

def parse_light_condition(p: Path):
    m = LIGHT_RE.match(p.name)
    return f'light_{int(m.group(1)):02d}' if m else None

test_img_dir = LOCAL_DATASET_DIR / 'images' / 'test'
test_lbl_dir = LOCAL_DATASET_DIR / 'labels' / 'test'

by_light = defaultdict(list)
unmatched = []
for p in test_img_dir.glob('*'):
    if p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
        continue
    cond = parse_light_condition(p)
    (by_light[cond] if cond else unmatched).append(p if cond else p.name)

for cond in sorted(by_light):
    print(f'{cond} -> {len(by_light[cond]):4d} images')
if unmatched:
    print(f'[warn] {len(unmatched)} files unparsed (e.g. {unmatched[:3]}); excluded.')
if not by_light:
    raise RuntimeError('No lighting conditions parsed - check the filename format.')

In [ ]:
# Build one tiny dataset per lighting condition (symlinks + a standalone data.yaml).
# train/val are placeholders pointing at images/test because Ultralytics requires both keys.
if LOCAL_ILLUM_ROOT.exists():
    shutil.rmtree(LOCAL_ILLUM_ROOT)

illum_yamls = {}
for cond, imgs in by_light.items():
    sub_root = LOCAL_ILLUM_ROOT / cond
    (sub_root / 'images' / 'test').mkdir(parents=True, exist_ok=True)
    (sub_root / 'labels' / 'test').mkdir(parents=True, exist_ok=True)

    for src_img in imgs:
        dst_img = sub_root / 'images' / 'test' / src_img.name
        if not dst_img.exists():
            try:    os.symlink(src_img, dst_img)
            except OSError: shutil.copy2(src_img, dst_img)
        src_lbl = test_lbl_dir / (src_img.stem + '.txt')
        if src_lbl.exists():
            dst_lbl = sub_root / 'labels' / 'test' / src_lbl.name
            if not dst_lbl.exists():
                try:    os.symlink(src_lbl, dst_lbl)
                except OSError: shutil.copy2(src_lbl, dst_lbl)

    yml_path = sub_root / 'data.yaml'
    with open(yml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump({
            'path' : str(sub_root),
            'train': 'images/test', 'val': 'images/test', 'test': 'images/test',
            'nc'   : len(CLASSES), 'names': CLASSES,
        }, f, sort_keys=False)
    illum_yamls[cond] = yml_path

print(f'Prepared {len(illum_yamls)} per-illumination subsets.')

In [ ]:
baseline = YOLO(str(BASELINE_BEST))
p2       = YOLO(str(P2_BEST))

In [ ]:
def metric_row(tag, m):
    d = m.results_dict
    p = d.get('metrics/precision(B)')
    r = d.get('metrics/recall(B)')
    f1 = (2 * p * r) / (p + r) if (p is not None and r is not None and p + r > 0) else None
    return {
        'exp'       : tag,
        'map50'     : d.get('metrics/mAP50(B)'),
        'map50_95'  : d.get('metrics/mAP50-95(B)'),
        'precision' : p,
        'recall'    : r,
        'f1'        : f1,
        'fnr'       : (1.0 - r) if r is not None else None,
    }

def per_class_ap50(m, classes):
    if not hasattr(m.box, 'ap50'):
        return {c: None for c in classes}
    arr = np.array(m.box.ap50)
    if arr.size != len(classes):
        return {c: None for c in classes}
    return {c: float(v) for c, v in zip(classes, arr)}

In [ ]:
rows = []
per_class_records = []

for cond, yml_path in sorted(illum_yamls.items()):
    print(f'>>> {cond}')
    for tag, model in [('baseline', baseline), ('p2', p2)]:
        m = model.val(data=str(yml_path), split='test', imgsz=IMG_SIZE,
                      device=DEVICE, verbose=False)
        row = metric_row(f'{tag}_{cond}', m)
        row['model'] = tag; row['light'] = cond
        rows.append(row)
        for cn, v in per_class_ap50(m, CLASSES).items():
            per_class_records.append({'model': tag, 'light': cond, 'class': cn, 'ap50': v})

df    = pd.DataFrame(rows)
df_pc = pd.DataFrame(per_class_records)
df.to_csv(OUT_DIR / 'illumination_results.csv', index=False)
df_pc.to_csv(OUT_DIR / 'illumination_per_class.csv', index=False)
pd.set_option('display.float_format', lambda v: f'{v:.4f}' if isinstance(v, float) else str(v))
print(df[['model', 'light', 'map50', 'map50_95', 'recall', 'fnr']])

In [ ]:
lights = sorted(by_light.keys())
x = np.arange(len(lights)); w = 0.38

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title, ylim in [
    (axes[0], 'map50', 'mAP@0.5 per illumination',             (0.0, 1.0)),
    (axes[1], 'fnr',   'False Negative Rate per illumination', (0.0, 0.5)),
]:
    b = [df[(df['model'] == 'baseline') & (df['light'] == L)][col].iloc[0] for L in lights]
    p = [df[(df['model'] == 'p2')       & (df['light'] == L)][col].iloc[0] for L in lights]
    ax.bar(x - w/2, b, w, label='Baseline')
    ax.bar(x + w/2, p, w, label='P2')
    ax.set_xticks(x); ax.set_xticklabels(lights, rotation=20)
    ax.set_title(title); ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3); ax.legend()
    for xi, v in zip(x - w/2, b):
        if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
    for xi, v in zip(x + w/2, p):
        if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

fig.suptitle('Multi-Illumination Robustness on Test Split')
fig.tight_layout()
fig.savefig(OUT_DIR / 'plots' / 'illumination_bars.png', dpi=150)
plt.show()

In [ ]:
for model_name in ['baseline', 'p2']:
    pivot = (df_pc[df_pc['model'] == model_name]
             .pivot(index='light', columns='class', values='ap50')
             .reindex(sorted(by_light))[CLASSES])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', vmin=0.0, vmax=1.0,
                cbar_kws={'label': 'AP@0.5'}, ax=ax)
    ax.set_title(f'AP@0.5 heatmap - {model_name}')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'plots' / f'illumination_heatmap_{model_name}.png', dpi=150)
    plt.show()

In [ ]:
summary = []
for model_name in ['baseline', 'p2']:
    sub = df[df['model'] == model_name]
    summary.append({
        'model'       : model_name,
        'mean_map50'  : float(sub['map50'].mean()),
        'min_map50'   : float(sub['map50'].min()),
        'max_map50'   : float(sub['map50'].max()),
        'std_map50'   : float(sub['map50'].std()),
        'worst_light' : sub.loc[sub['map50'].idxmin(), 'light'],
        'best_light'  : sub.loc[sub['map50'].idxmax(), 'light'],
        'mean_fnr'    : float(sub['fnr'].mean()),
        'max_fnr'     : float(sub['fnr'].max()),
    })
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / 'illumination_summary.csv', index=False)
print(summary_df)

In [ ]:
(OUT_DIR / 'illumination_summary.json').write_text(json.dumps({
    'p2_aug_profile'  : P2_AUG_PROFILE,
    'lights_found'    : sorted(by_light),
    'images_per_light': {k: len(v) for k, v in by_light.items()},
    'rows'            : rows,
    'summary'         : summary,
}, indent=2, default=str), encoding='utf-8')
print('Outputs in:', OUT_DIR)